In [ ]:
%pip install -q -U "crewai[google-genai]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.4/724.4 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import json
from google.colab import userdata

from crewai import Agent, Task, Crew, LLM
from crewai.flow.flow import Flow, start, listen, router, or_
from pydantic import BaseModel, Field

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "Add GEMINI_API_KEY in Google Colab Secrets."
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

llm = LLM(
    model="gemini/gemini-3.1-flash-lite-preview",
    api_key=GEMINI_API_KEY,
    temperature=0.1
)

print("Gemini configured successfully!")

Gemini configured successfully!


In [ ]:
class BugTriageState(BaseModel):
    issue_title: str = ""
    issue_body: str = ""
    issue_type: str = "question"
    classification_reason: str = ""
    result: dict = Field(default_factory=dict)

In [ ]:
# Get bug details from the user
title = input("Enter the bug title: ").strip()
body = input("Describe the bug in detail: ").strip()

if not title or not body:
    print("Please provide both a bug title and description.")
else:
    sample_issue = {
        "title": title,
        "body": body
    }

    print("\nBug report received:")
    print("Title:", sample_issue["title"])
    print("Description:", sample_issue["body"])

Sample issue loaded:
Application crashes when uploading a PDF


In [ ]:
class BugTriageFlow(Flow[BugTriageState]):

    @start()
    def read_issue(self):
        print("\n📥 Reading issue...")

        self.state.issue_title = sample_issue["title"]
        self.state.issue_body = sample_issue["body"]

        print("Title:", self.state.issue_title)
        return "issue_loaded"

    @listen(read_issue)
    def classify_issue(self, _):
        print("\n🔍 Classifying issue...")

        prompt = f"""
Classify the following software issue into exactly one category:
bug, feature, or question.

Title:
{self.state.issue_title}

Body:
{self.state.issue_body}

Rules:
- bug: existing behavior is broken, incorrect, or crashes.
- feature: requests new or enhanced functionality.
- question: asks for information or help without reporting
  a software defect or requesting a feature.

Return ONLY valid JSON:
{{
  "issue_type": "bug",
  "reason": "Brief explanation"
}}
"""

        response = llm.call(prompt)

        try:
            raw = str(response).strip()
            raw = raw.replace("```json", "").replace("```", "").strip()
            data = json.loads(raw)

            issue_type = str(data["issue_type"]).lower().strip()

            if issue_type not in {"bug", "feature", "question"}:
                raise ValueError("Invalid issue category")

            self.state.issue_type = issue_type
            self.state.classification_reason = str(data["reason"])

        except (ValueError, TypeError, KeyError, json.JSONDecodeError):
            self.state.issue_type = "question"
            self.state.classification_reason = (
                "Classification could not be parsed; "
                "manual review is required."
            )

        print("Category:", self.state.issue_type)
        print("Reason:", self.state.classification_reason)

        return self.state.issue_type

    @router(classify_issue)
    def route_type(self):
        category = self.state.issue_type

        print(f"\n➡️ Routing to: {category}")

        if category == "bug":
            return "bug"
        elif category == "feature":
            return "feature"
        else:
            return "question"

    @listen("bug")
    def bug_crew(self):
        agent = Agent(
            role="Bug Triage Specialist",
            goal="Analyze bugs and suggest reproduction steps",
            backstory=(
                "You help software teams understand defects, "
                "severity, and steps to reproduce them."
            ),
            llm=llm,
            verbose=False
        )

        task = Task(
            description=f"""
Analyze this reported bug.

Title: {self.state.issue_title}
Body: {self.state.issue_body}

Classification reason: {self.state.classification_reason}

Provide:
1. Suggested severity: low, medium, high, or critical
2. Steps to reproduce based on the supplied information
3. Expected versus actual behavior
4. Missing information that needs verification

Do not invent technical details.
""",
            expected_output="A structured bug triage report.",
            agent=agent
        )

        result = str(Crew(
            agents=[agent],
            tasks=[task],
            verbose=False
        ).kickoff())

        self.state.result = {
            "category": "bug",
            "title": self.state.issue_title,
            "analysis": result
        }

        print("\n🐛 Bug analysis completed.")
        return "bug_completed"

    @listen("feature")
    def feature_crew(self):
        agent = Agent(
            role="Product Analyst",
            goal="Convert feature requests into clear user stories",
            backstory=(
                "You help product teams define user needs "
                "and acceptance criteria."
            ),
            llm=llm,
            verbose=False
        )

        task = Task(
            description=f"""
Turn this feature request into a draft user story.

Title: {self.state.issue_title}
Body: {self.state.issue_body}

Include:
1. User story in the format As a..., I want..., so that...
2. Acceptance criteria
3. Clarifying questions

Do not invent requirements that are not supported by the issue.
""",
            expected_output="A user story with acceptance criteria.",
            agent=agent
        )

        result = str(Crew(
            agents=[agent],
            tasks=[task],
            verbose=False
        ).kickoff())

        self.state.result = {
            "category": "feature",
            "title": self.state.issue_title,
            "analysis": result
        }

        print("\n💡 Feature story drafted.")
        return "feature_completed"

    @listen("question")
    def answer_crew(self):
        agent = Agent(
            role="Issue Support Assistant",
            goal="Draft a helpful response to the reported question",
            backstory=(
                "You help users understand software behavior "
                "without inventing undocumented facts."
            ),
            llm=llm,
            verbose=False
        )

        task = Task(
            description=f"""
Draft a concise, helpful response to this issue.

Title: {self.state.issue_title}
Body: {self.state.issue_body}

If documentation or technical details are missing, clearly
state what information is needed. Do not invent product facts.
""",
            expected_output="A clear draft support response.",
            agent=agent
        )

        result = str(Crew(
            agents=[agent],
            tasks=[task],
            verbose=False
        ).kickoff())

        self.state.result = {
            "category": "question",
            "title": self.state.issue_title,
            "analysis": result
        }

        print("\n💬 Support response drafted.")
        return "question_completed"

    @listen(or_(bug_crew, feature_crew, answer_crew))
    def post_reply(self, _):
        print("\n📌 Preparing issue response...")

        self.state.result["classification_reason"] = (
            self.state.classification_reason
        )
        self.state.result["reply_status"] = "Draft ready for human review"

        output_path = "/content/bug_triage_result.json"

        with open(output_path, "w", encoding="utf-8") as file:
            json.dump(
                self.state.result,
                file,
                indent=2,
                ensure_ascii=False
            )

        print("Category:", self.state.result["category"])
        print("Saved:", output_path)
        print("Reply prepared for human review.")

        return output_path

In [ ]:
flow = BugTriageFlow()
flow.kickoff()

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: BugTriageFlow                                                                                            │
│  ID: b1fe459f-3cfd-4456-8c4e-45ca709b999f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: BugTriageFlow                                                                                            │
│  ID: b1fe459f-3cfd-4456-8c4e-45ca709b999f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


📥 Reading issue...
Title: Application crashes when uploading a PDF


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: read_issue                                                                                             │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


🔍 Classifying issue...


...


📌 Preparing issue response...
Category: bug
Saved: /content/bug_triage_result.json
Reply prepared for human review.


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: post_reply                                                                                             │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: post_reply                                                                                             │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: BugTriageFlow                                                                                            │
│  ID: b1fe459f-3cfd-4456-8c4e-45ca709b999f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

'/content/bug_triage_result.json'

In [ ]:
print(json.dumps(flow.state.result, indent=2, ensure_ascii=False))

{
  "category": "bug",
  "title": "Application crashes when uploading a PDF",
  "analysis": "**Bug Triage Report**\n\n**1. Suggested Severity:**\n* **High** \n*(Reasoning: The application completely crashes rather than handling the error gracefully, which impacts core functionality and disrupts the user experience for a standard use case—uploading large files.)*\n\n**2. Steps to Reproduce:**\n1. Log in to the application.\n2. Navigate to the file upload section.\n3. Select a PDF file that is larger than 10 MB.\n4. Attempt to upload the file.\n5. Observe the application's response.\n\n**3. Expected versus Actual Behavior:**\n* **Expected Behavior:** The application should either successfully upload the file or display a clear, user-friendly error message indicating that the file size exceeds the allowed limit.\n* **Actual Behavior:** The application crashes entirely when attempting to upload a PDF larger than 10 MB.\n\n**4. Missing Information that Needs Verification:**\n* Exact browser

In [ ]:
from google.colab import files

files.download("/content/bug_triage_result.json")